# 재활용 이미지 VQA 파인튜닝 (개선 버전)

베이스라인 대비 주요 개선 사항:
- **모델 업그레이드**: Qwen2.5-VL-3B → **7B** (A100 GPU 활용)
- **전체 데이터 활용**: 200개 → **전체 학습 데이터** (train + human_check + dev 최빈값)
- **고품질 데이터 우선**: human_check 검수 데이터를 dev보다 우선 적용
- **프롬프트 개선**: 재활용 분류 도메인 전문 시스템 프롬프트 + 한국어 강화 지시
- **LoRA 강화**: r=8 → r=16 (더 많은 학습 파라미터)
- **2 에포크 학습**: 1 → 2 에포크 (정확도 향상)
- **코사인 스케줄러**: linear → cosine with warmup (학습 안정성)
- **강건한 답변 추출**: 정규식 기반 다중 fallback 파서


# 환경 준비

개발 환경에 필요한 라이브러리를 설치합니다.

- A100 GPU 기준 최적화 (CUDA 12.1)
- 실행 완료 후 **런타임 → 세션 다시 시작** 필요


In [3]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# GPU / 라이브러리 버전 확인


In [39]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB" if torch.cuda.is_available() else "")


Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


# 데이터 준비

구글 드라이브를 마운트하고 데이터를 불러옵니다.

사용 데이터:
- `train.csv` + `train/` 폴더 → 5,073개 (정제된 정답)
- `human_check.csv` → 668개 (검수자가 직접 검토한 고품질 dev 데이터)
- `dev.csv` + `dev/` 폴더 → 4,413개에서 human_check 제외 후 최빈값 사용
- `test.csv` + `test/` 폴더 → 5,074개 (추론 대상)

#### ColabNotebooks/content 폴더 경로 사용


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import urllib.request
import os

os.makedirs("/content/data", exist_ok=True)

url = "https://raw.githubusercontent.com/mo-gun/SSAFYAI/main/human_check.csv"
save_path = "/content/data/human_check.csv"

if not os.path.exists(save_path):
    urllib.request.urlretrieve(url, save_path)

print("human_check.csv downloaded")

human_check.csv downloaded


In [13]:
import os

# ─── 경로 설정 ───────────────────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/content"
# GitHub에서 human_check 불러오기
HC_CSV = "https://raw.githubusercontent.com/mo-gun/SSAFYAI/main/human_check.csv"

TRAIN_CSV  = os.path.join(BASE_DIR, 'train.csv')
DEV_CSV    = os.path.join(BASE_DIR, 'dev.csv')
# HC_CSV     = os.path.join(BASE_DIR, 'human_check.csv')
TEST_CSV   = os.path.join(BASE_DIR, 'test.csv')
SUB_CSV    = os.path.join(BASE_DIR, 'sample_submission.csv')

TRAIN_IMG  = os.path.join(BASE_DIR, 'train')
DEV_IMG    = os.path.join(BASE_DIR, 'dev')
TEST_IMG   = os.path.join(BASE_DIR, 'test')

# 저장 디렉토리
OUTPUT_DIR = '/content/qwen25vl_7b_recycling'
SUB_SAVE   = '/content/submission.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ 경로 설정 완료")


✅ 경로 설정 완료


# 라이브러리, 모델, 전역 설정

- **모델**: `Qwen/Qwen2.5-VL-7B-Instruct` (7B 파라미터, A100 최적화)
- **IMAGE_SIZE**: 448px (베이스라인 384 → 448, 더 정밀한 이미지 인식)
- **SEED**: 재현성 고정


In [68]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# ─── 전역 하이퍼파라미터 ─────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-VL-7B-Instruct"  # 7B 모델 (베이스라인 3B → 7B)
IMAGE_SIZE     = 448   # 해상도 업: 384 → 448
MAX_NEW_TOKENS = 8
SEED           = 42
NUM_EPOCHS     = 2     # 학습 에포크: 1 → 2
#---시간줄이기--
# NUM_EPOCHS     = 1
LR             = 2e-4
# GRAD_ACCUM     = 4     # 유효 배치 크기 = 1 × 8 = 8
GRAD_ACCUM     = 2     # 유효 배치 크기 = 1 × 8 = 8
WARMUP_RATIO   = 0.05
LORA_R         = 16    # LoRA rank: 8 → 16
LORA_ALPHA     = 32

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("✅ 설정 완료")


Device: cuda
✅ 설정 완료


# 데이터 로드 및 병합

세 가지 데이터소스를 병합하여 최대한 많은 고품질 학습 데이터를 구성합니다.

| 데이터 | 개수 | 신뢰도 | 처리 방법 |
|--------|------|--------|----------|
| `train.csv` | 5,073 | ★★★ | 그대로 사용 |
| `human_check.csv` | 668 | ★★★ | `answer` 컬럼 사용 |
| `dev.csv` (나머지) | ~3,745 | ★★ | 5명 응답 **최빈값** 사용 |

**human_check 데이터는 dev보다 우선 적용** (중복 ID 제거)


In [69]:
# ─── train.csv 로드 ─────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
train_df['path'] = train_df['path'].apply(
    lambda p: os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p
)

# ─── human_check.csv 로드 (검수자 정답 우선 사용) ───────────────────────
hc_df = pd.read_csv(HC_CSV)
hc_df['path'] = hc_df['path'].apply(
    lambda p: os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p
)
hc_df = hc_df[['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer']].copy()

# ─── dev.csv 로드 + 최빈값 계산 ─────────────────────────────────────────
dev_df = pd.read_csv(DEV_CSV)
dev_df['path'] = dev_df['path'].apply(
    lambda p: os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p
)
# 5명 응답 최빈값 (동점 시 첫 번째)
ans_cols = ['answer1', 'answer2', 'answer3', 'answer4', 'answer5']
dev_df['answer'] = dev_df[ans_cols].mode(axis=1)[0]

# human_check에 포함된 ID는 dev에서 제거 (human_check 우선)
hc_ids = set(hc_df['id'])
dev_only_df = dev_df[~dev_df['id'].isin(hc_ids)][['id','path','question','a','b','c','d','answer']].copy()

# ─── 전체 학습 데이터 병합 ───────────────────────────────────────────────
all_train_df = pd.concat([train_df, hc_df, dev_only_df], ignore_index=True)
all_train_df = all_train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# ─── 학습 데이터 절반만 사용 (속도 테스트용) ─────────────────────────
all_train_df = all_train_df.sample(frac=0.5, random_state=SEED).reset_index(drop=True)


# ─── test.csv 로드 ───────────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
test_df['path'] = test_df['path'].apply(
    lambda p: os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p
)

print(f"train.csv:       {len(train_df):>5}개")
print(f"human_check.csv: {len(hc_df):>5}개  (검수 고품질)")
print(f"dev (나머지):    {len(dev_only_df):>5}개  (최빈값)")
print(f"─────────────────────────────")
print(f"학습 데이터 합계: {len(all_train_df):>5}개")
print(f"테스트 데이터:    {len(test_df):>5}개")
print("\n답변 분포:")
print(all_train_df['answer'].value_counts().sort_index())


train.csv:        5073개
human_check.csv:   668개  (검수 고품질)
dev (나머지):     3745개  (최빈값)
─────────────────────────────
학습 데이터 합계:  4743개
테스트 데이터:     5074개

답변 분포:
answer
a    1212
b    1196
c    1130
d    1205
Name: count, dtype: int64


# 모델, Processor 로드

- **4-bit NF4 양자화**: VRAM 절약 (7B 모델 ≈ 7GB)
- **LoRA rank=16**: 베이스라인 대비 2배 → 더 많은 파라미터 업데이트
- **타겟 모듈**: 어텐션 + FFN 전체 (q/k/v/o + gate/up/down proj)

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()


In [70]:
# ─── 4-bit 양자화 설정 ──────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100은 bfloat16 지원
)

# ─── Processor 로드 ──────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# ─── 사전학습 모델 로드 (7B) ─────────────────────────────────────────────
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# ─── 양자화 + 그래디언트 체크포인팅 ─────────────────────────────────────
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# ─── LoRA 설정 (rank=16, alpha=32) ───────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706


# 프롬프트 템플릿

베이스라인 구조를 유지하면서 재활용 분류 도메인에 최적화된 프롬프트를 적용합니다.

**개선 포인트**:
1. **시스템 프롬프트**: 재활용 분류 전문가 역할 + 재질/수량/종류 판단 강조
2. **유저 프롬프트**: 한국어 지시 강화 + 이미지 주의사항 명시
3. **One-shot 없이 zero-shot**: 이미지마다 맥락이 달라 직접 판단 유도

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()


In [71]:
# ─── 시스템 지시사항 ─────────────────────────────────────────────────────
SYSTEM_INSTRUCT = (
    "You are a Korean recycling waste classification expert. "
    "You carefully examine images showing recyclable items such as plastic bottles, "
    "glass bottles, cans, paper, and their surrounding environment. "
    "For each image, you analyze the material type, quantity, and category of recycling items. "
    "Answer the Korean multiple-choice question by outputting exactly ONE lowercase letter "
    "from: a, b, c, or d. Output only the letter — no explanation, no punctuation."
)

# ─── 사용자 프롬프트 빌더 ────────────────────────────────────────────────
def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    """
    재활용 이미지 VQA 전용 객관식 프롬프트.
    - 이미지를 꼼꼼히 보도록 유도
    - 재질/수량/종류에 집중하도록 안내
    - 반드시 소문자 한 글자(a/b/c/d)만 출력하도록 강제
    """
    return (
        "이미지를 꼼꼼히 살펴보고 아래 질문에 답하세요.\n"
        "재활용 품목의 재질, 종류, 수량을 정확히 파악하는 것이 중요합니다.\n\n"
        f"[질문] {question}\n\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "위 선택지 중 가장 정확한 답의 알파벳 소문자 한 글자만 출력하세요. (a, b, c, d 중 하나)"
    )

# 동작 확인
sample_prompt = build_mc_prompt("사진 속 재활용 가능한 병의 종류는?", "금속 캔", "유리 병", "플라스틱 병", "종이 팩")
print("─── 프롬프트 예시 ───")
print(sample_prompt)


─── 프롬프트 예시 ───
이미지를 꼼꼼히 살펴보고 아래 질문에 답하세요.
재활용 품목의 재질, 종류, 수량을 정확히 파악하는 것이 중요합니다.

[질문] 사진 속 재활용 가능한 병의 종류는?

(a) 금속 캔
(b) 유리 병
(c) 플라스틱 병
(d) 종이 팩

위 선택지 중 가장 정확한 답의 알파벳 소문자 한 글자만 출력하세요. (a, b, c, d 중 하나)


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝
    - IntentDataset()


In [72]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')

        q = str(row['question'])
        a, b, c, d = str(row['a']), str(row['b']), str(row['c']), str(row['d'])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': user_text}
            ]}
        ]
        if self.train:
            gold = str(row['answer']).strip().lower()
            messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': gold}]})

        return {'messages': messages, 'image': img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample['messages'],
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample['image'])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors='pt'
        )
        if self.train:
            enc['labels'] = enc['input_ids'].clone()
        return enc


# DataLoader

- 학습/검증 분리: 95% / 5% (전체 데이터가 많으므로 검증 비율 축소)
- 배치 크기 1 + Gradient Accumulation 8 = 유효 배치 8

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()


In [73]:
# ─── 학습/검증 분리 (95 / 5) ────────────────────────────────────────────
split = int(len(all_train_df) * 0.95)
train_subset = all_train_df.iloc[:split]
valid_subset = all_train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True,
    collate_fn=DataCollator(processor, True), num_workers=4
)
valid_loader = DataLoader(
    valid_ds, batch_size=4, shuffle=False,
    collate_fn=DataCollator(processor, True), num_workers=4
)

print(f"학습 셋:  {len(train_ds):>5}개")
print(f"검증 셋:  {len(valid_ds):>5}개")


학습 셋:   4505개
검증 셋:    238개


# Fine-tuning

- **에포크**: 2 (베이스라인 1 → 2)
- **학습률**: 2e-4 + **코사인 스케줄러** (선형 → 코사인, 학습 후반 안정화)
- **Warmup**: 전체 스텝의 5%
- **Gradient Accumulation**: 8 스텝 (유효 배치 8)
- **Mixed Precision**: bfloat16 (A100 최적)
- **Gradient Clipping**: max_norm=1.0 (학습 안정화)

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6


In [74]:
model = model.to(device)


# ─── 옵티마이저 ──────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01
)

# ─── 코사인 스케줄러 (warmup 포함) ───────────────────────────────────────
num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps   = int(num_training_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps, num_training_steps
)

# ─── AMP 스케일러 ────────────────────────────────────────────────────────
scaler = torch.cuda.amp.GradScaler(enabled=True)

best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # ── 학습 ──────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', unit='batch')

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            # 그래디언트 클리핑 (학습 안정화)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg = running_loss / GRAD_ACCUM
            progress_bar.set_postfix({'loss': f'{avg:.3f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
            running_loss = 0.0

    # ── 검증 ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [valid]', unit='batch'):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val = val_loss / val_steps
    print(f'[Epoch {epoch+1}] val_loss={avg_val:.4f}')

    # 최고 성능 모델 저장
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f'  ✅ Best model saved → {OUTPUT_DIR}  (val_loss={avg_val:.4f})')

print("\n🏁 학습 완료")


/tmp/ipykernel_3844/3519158394.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epoch 1/2 [train]:   0%|          | 0/1127 [00:00<?, ?batch/s]

/tmp/ipykernel_3844/3519158394.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipykernel_3844/3519158394.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):


Epoch 1/2 [valid]:   0%|          | 0/60 [00:00<?, ?batch/s]

[Epoch 1] val_loss=3.8320
  ✅ Best model saved → /content/qwen25vl_7b_recycling  (val_loss=3.8320)


Epoch 2/2 [train]:   0%|          | 0/1127 [00:00<?, ?batch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78c065566e80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78c065566e80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 2/2 [valid]:   0%|          | 0/60 [00:00<?, ?batch/s]

[Epoch 2] val_loss=3.8283
  ✅ Best model saved → /content/qwen25vl_7b_recycling  (val_loss=3.8283)

🏁 학습 완료


# Inference

- **답변 파서**: 정규식 기반 다중 fallback (베이스라인 대비 강건성 향상)
  1. 텍스트 마지막 줄에서 단일 알파벳 추출
  2. 전체 텍스트에서 `(a)` / `답: a` 패턴 탐색
  3. 어디에도 없으면 `a` 반환 (최빈값 fallback)
- **최고 성능 모델**로 추론 (best checkpoint 로드)

#### 실습 참고 내용

    챕터 4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 추론 : with torch.no_grad(), model.eval()


In [75]:
# ─── 강건한 답변 파서 ────────────────────────────────────────────────────
def extract_choice(text: str) -> str:
    """
    모델 응답에서 a/b/c/d 중 하나를 추출합니다.
    다중 fallback으로 베이스라인 대비 파싱 정확도를 높입니다.
    """
    text = text.strip().lower()
    valid = {'a', 'b', 'c', 'd'}

    # 1) 마지막 줄의 단일 알파벳
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines and lines[-1] in valid:
        return lines[-1]

    # 2) 토큰 단위 탐색 (마지막 줄)
    if lines:
        for tok in reversed(lines[-1].split()):
            tok = tok.strip('.,!?;:()')
            if tok in valid:
                return tok

    # 3) 정규식: '정답: a', '답변: b', '(c)', 'answer: d' 패턴
    patterns = [
        r'(?:정답|답변|answer|ans)[\s:：=]+([abcd])',
        r'\(([abcd])\)',
        r'\b([abcd])\b',
    ]
    for pat in patterns:
        matches = re.findall(pat, text)
        if matches:
            return matches[-1]   # 마지막 매칭

    # 4) 최후 fallback
    return 'a'


# ─── 추론 (테스트 데이터 전체) ───────────────────────────────────────────
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc='Inference', unit='sample'):
    row = test_df.iloc[i]
    img = Image.open(row['path']).convert('RGB')
    user_text = build_mc_prompt(
        str(row['question']),
        str(row['a']), str(row['b']), str(row['c']), str(row['d'])
    )

    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': user_text}
        ]}
    ]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_input], images=[img], return_tensors='pt'
    ).to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

print("\n추론 완료!")
print("예측 분포:", pd.Series(preds).value_counts().sort_index().to_dict())


Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

/tmp/ipykernel_3844/2598173974.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [77]:
# ─────────────────────────────────────────────────────────
# tokenizer padding 설정 (decoder-only 경고 해결)
# ─────────────────────────────────────────────────────────

processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


# ─── 강건한 답변 파서 ─────────────────────────────────────
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    valid = {'a', 'b', 'c', 'd'}

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines and lines[-1] in valid:
        return lines[-1]

    if lines:
        for tok in reversed(lines[-1].split()):
            tok = tok.strip('.,!?;:()')
            if tok in valid:
                return tok

    patterns = [
        r'(?:정답|답변|answer|ans)[\s:：=]+([abcd])',
        r'\(([abcd])\)',
        r'\b([abcd])\b',
    ]

    for pat in patterns:
        matches = re.findall(pat, text)
        if matches:
            return matches[-1]

    return 'a'


# ─── 추론 (Batch Inference) ──────────────────────────────

BATCH_SIZE = 8
MAX_NEW_TOKENS = 5

model.eval()
preds = []

for start_idx in tqdm(range(0, len(test_df), BATCH_SIZE), desc='Inference', unit='batch'):

    batch_df = test_df.iloc[start_idx:start_idx + BATCH_SIZE]

    images = []
    texts = []

    for _, row in batch_df.iterrows():

        img = Image.open(row['path']).convert('RGB')
        img.load()

        images.append(img)

        user_text = build_mc_prompt(
            str(row['question']),
            str(row['a']),
            str(row['b']),
            str(row['c']),
            str(row['d'])
        )

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': user_text}
            ]}
        ]

        text_input = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text_input)

    inputs = processor(
        text=texts,
        images=images,
        return_tensors='pt',
        padding=True
    ).to(device)

    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):

        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    outputs = processor.batch_decode(out_ids, skip_special_tokens=True)

    for out in outputs:
        preds.append(extract_choice(out))


print("\n추론 완료!")
print("예측 분포:", pd.Series(preds).value_counts().sort_index().to_dict())

Inference:   0%|          | 0/635 [00:00<?, ?batch/s]


추론 완료!
예측 분포: {'a': 925, 'b': 1358, 'c': 1321, 'd': 1470}


# 제출 파일 생성

`sample_submission.csv`와 동일한 형식으로 저장합니다.
- 컬럼: `id`, `answer`
- `answer`: a, b, c, d 중 소문자 한 글자


In [78]:
# ─── 제출 파일 저장 ──────────────────────────────────────────────────────
submission = pd.DataFrame({
    'id':     test_df['id'].values,
    'answer': preds
})

# 형식 검증
assert len(submission) == len(test_df), '행 수 불일치!'
assert submission['answer'].isin(['a','b','c','d']).all(), '잘못된 답변 값 존재!'

submission.to_csv(SUB_SAVE, index=False)
print(f"✅ 제출 파일 저장 완료: {SUB_SAVE}")
print(f'총 {len(submission)}개 예측')
print()
print('─── 앞 5개 확인 ───')
print(submission.head())
print()
print('─── 정답 분포 ───')
print(submission['answer'].value_counts().sort_index())

from google.colab import files
files.download(SUB_SAVE)


✅ 제출 파일 저장 완료: /content/submission.csv
총 5074개 예측

─── 앞 5개 확인 ───
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      b
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      d

─── 정답 분포 ───
answer
a     925
b    1358
c    1321
d    1470
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# (선택) 추론 예시 확인

모델이 실제로 어떤 텍스트를 생성하는지 확인합니다.


In [ ]:
# ─── 임의 샘플 5개 추론 결과 출력 ──────────────────────────────────────
sample_idxs = random.sample(range(len(test_df)), min(5, len(test_df)))

for idx in sample_idxs:
    row = test_df.iloc[idx]
    img = Image.open(row['path']).convert('RGB')
    user_text = build_mc_prompt(
        str(row['question']),
        str(row['a']), str(row['b']), str(row['c']), str(row['d'])
    )
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': user_text}
        ]}
    ]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text_input], images=[img], return_tensors='pt').to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    raw = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    pred = extract_choice(raw)

    print(f'ID: {row["id"]}')
    print(f'  질문: {row["question"]}')
    print(f'  선택지: a={row["a"]}  b={row["b"]}  c={row["c"]}  d={row["d"]}')
    print(f'  모델 원본 출력: {repr(raw[-50:])}')
    print(f'  최종 예측: [{pred}]')
    print()
